# Unified Baseline Notebook

- `Train Mean`
- `Last Observation`
- `RF Only`

It is set up to run the three symptom targets in one pass:

- `Negative`
- `Cognitive`
- `Positive`


In [1]:
import json
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

TARGET_MAP = {
    "Negative": 0,
    "Cognitive": 1,
    "Positive": 2,
}

DEFAULT_RF_GRID = [
    {
        "n_estimators": 300,
        "max_depth": 4,
        "min_samples_leaf": 4,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 300,
        "max_depth": 8,
        "min_samples_leaf": 4,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 400,
        "max_depth": None,
        "min_samples_leaf": 8,
        "max_features": 0.2,
    },
]

LOCAL_DATA_ROOT = Path(
    "/Users/f007qrc/Library/CloudStorage/"
    "GoogleDrive-anna.m.langener@dartmouth.edu/.shortcut-targets-by-id/"
    "1_T-CzrkNtF9QAhex6qrEOA0ZQIL7g5u5/"
    "SensingIndividualized/deeplearning_data/pred"
)

COLAB_DATA_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/"
    "Anna_JacobsonColabDrive/SensingIndividualized/deeplearning_data/pred"
)

LOCAL_PRECOMPUTED_EMBEDDING_ROOT = Path(
    "/Users/f007qrc/Library/CloudStorage/"
    "GoogleDrive-anna.m.langener@dartmouth.edu/.shortcut-targets-by-id/"
    "1_T-CzrkNtF9QAhex6qrEOA0ZQIL7g5u5/"
    "SensingIndividualized/embeddings_pred"
)

COLAB_PRECOMPUTED_EMBEDDING_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/"
    "Anna_JacobsonColabDrive/SensingIndividualized/embeddings_pred"
)

PRECOMPUTED_EMBEDDING_ROOT = LOCAL_PRECOMPUTED_EMBEDDING_ROOT

OUTPUT_ROOT = Path.cwd() / "baseline_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [79]:
def participant_id_candidates(participant_id):
    raw = str(participant_id)
    candidates = [raw]

    if raw.isdigit():
        num = int(raw)
        candidates.extend([f"{num:05d}", f"{num:06d}", str(num)])

    seen = set()
    ordered = []
    for candidate in candidates:
        if candidate not in seen:
            ordered.append(candidate)
            seen.add(candidate)
    return ordered


def resolve_participant_npz(participant_id, suffix="_4h.npz"):
    roots = [LOCAL_DATA_ROOT, COLAB_DATA_ROOT]

    for candidate in participant_id_candidates(participant_id):
        filename = f"participant_{candidate}{suffix}"
        for root in roots:
            path = root / filename
            if path.exists():
                return path

    searched = [str(root / f"participant_<candidate>{suffix}") for root in roots]
    raise FileNotFoundError(
        f"Could not find participant file for {participant_id}. Searched roots: {searched}"
    )


def load_participant_bundle(participant_id, drop_first_sample=False):
    path = resolve_participant_npz(participant_id)
    loaded = np.load(path, allow_pickle=True)

    timestamp_key = "datetimes" if "datetimes" in loaded.files else "dates"
    datetimes = pd.to_datetime(loaded[timestamp_key])
    labels_all = np.round(loaded["labels"]).astype(np.float32) / 100.0

    bundle = {
        "participant_id": str(participant_id),
        "path": path,
        "datetimes": datetimes,
        "labels_all": labels_all,
        "combined": loaded["combined"],
    }

    order = np.argsort(bundle["datetimes"].values)

    bundle["combined"] = bundle["combined"][order]
    bundle["labels_all"] = bundle["labels_all"][order]
    bundle["datetimes"] = bundle["datetimes"][order]

    if drop_first_sample:
        bundle["combined"] = bundle["combined"][1:]
        bundle["labels_all"] = bundle["labels_all"][1:]
        bundle["datetimes"] = bundle["datetimes"][1:]

    return bundle


def resolve_embedding_participant_dir(
    participant_id,
    embedding_root=None,
):
    if embedding_root is None:
        roots = [
            LOCAL_PRECOMPUTED_EMBEDDING_ROOT,
            COLAB_PRECOMPUTED_EMBEDDING_ROOT,
        ]
    else:
        roots = [Path(embedding_root)]

    for root in roots:
        if root.name == "shared" and root.exists():
            return root

        if root.name.startswith("participant_"):
            shared_path = root / "shared"
            if shared_path.exists():
                return shared_path
            if root.exists():
                return root

        for candidate in participant_id_candidates(participant_id):
            participant_root = root / f"participant_{candidate}"
            shared_path = participant_root / "shared"

            if shared_path.exists():
                return shared_path
            if participant_root.exists():
                return participant_root

    searched = [str(Path(root)) for root in roots]
    raise FileNotFoundError(
        f"No precomputed embedding directory found for participant {participant_id}. "
        f"Searched roots: {searched}"
    )


def load_embedding_tensor(path):
    import torch

    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )


def load_precomputed_embedding_tensor(
    base_dir,
    split_prefix,
    expected_n=None,
):
    base_dir = Path(base_dir)
    path = base_dir / f"{split_prefix}_combined.pt"

    if not path.exists():
        raise FileNotFoundError(
            f"Missing embedding file: {path}"
        )

    tensor = load_embedding_tensor(path)
    x = np.asarray(tensor, dtype=np.float32)

    if expected_n is not None and x.shape[0] != expected_n:
        raise ValueError(
            f"Embedding row mismatch for {path}: "
            f"expected {expected_n}, got {x.shape[0]}"
        )

    return x


def pool_precomputed_embeddings(
    tensor,
    strategy="mean_reg",
):
    x = np.asarray(tensor, dtype=np.float32)

    if x.ndim == 4 and strategy != "flatten":
        x = np.nanmean(x, axis=1)

    if x.ndim == 2:
        pooled = x

    elif x.ndim == 3:
        if strategy == "mean":
            pooled = np.nanmean(x, axis=1)
        elif strategy == "reg":
            pooled = x[:, -1, :]
        elif strategy == "mean_reg":
            pooled = np.concatenate(
                [
                    np.nanmean(x, axis=1),
                    x[:, -1, :],
                ],
                axis=1,
            )
        elif strategy == "flatten":
            pooled = x.reshape(x.shape[0], -1)
        else:
            raise ValueError(
                f"Unknown embedding pooling strategy: {strategy}"
            )

    elif x.ndim == 4 and strategy == "flatten":
        pooled = x.reshape(x.shape[0], -1)

    else:
        raise ValueError(
            f"Unexpected precomputed embedding shape: {x.shape}"
        )

    return ensure_2d_feature_matrix(pooled)


def load_precomputed_feature_matrix(
    base_dir,
    split_prefix,
    expected_n=None,
):
    tensor = load_precomputed_embedding_tensor(
        base_dir,
        split_prefix,
        expected_n=expected_n,
    )
    features = pool_precomputed_embeddings(tensor)

    return features


def load_participant_embedding_tensor(
    participant_id,
    outer_fold_idx,
    split_prefix,
    embedding_root=None,
    expected_n=None,
    inner_fold_idx=None,
):
    shared_dir = resolve_embedding_participant_dir(
        participant_id,
        embedding_root=embedding_root,
    )

    if inner_fold_idx is None:
        split_dir = shared_dir / f"outer{outer_fold_idx}" / "final"
    else:
        split_dir = (
            shared_dir
            / f"outer{outer_fold_idx}"
            / f"inner{inner_fold_idx}"
        )

    return load_precomputed_embedding_tensor(
        split_dir,
        split_prefix,
        expected_n=expected_n,
    )


def load_participant_embedding_split(
    participant_id,
    outer_fold_idx,
    split_prefix,
    embedding_root=None,
    expected_n=None,
    inner_fold_idx=None,
):
    shared_dir = resolve_embedding_participant_dir(
        participant_id,
        embedding_root=embedding_root,
    )

    if inner_fold_idx is None:
        split_dir = shared_dir / f"outer{outer_fold_idx}" / "final"
    else:
        split_dir = (
            shared_dir
            / f"outer{outer_fold_idx}"
            / f"inner{inner_fold_idx}"
        )

    return load_precomputed_feature_matrix(
        split_dir,
        split_prefix,
        expected_n=expected_n,
    )

def select_target(bundle, target_name):
    if target_name not in TARGET_MAP:
        raise ValueError(
            f"Unknown target: {target_name}. "
            f"Valid targets are: {list(TARGET_MAP.keys())}"
        )

    target_idx = TARGET_MAP[target_name]

    labels_all = np.asarray(
        bundle["labels_all"],
        dtype=np.float32,
    )

    if labels_all.ndim != 2:
        raise ValueError(
            f"Expected labels_all to have shape [N, targets], "
            f"but received {labels_all.shape}"
        )

    if target_idx >= labels_all.shape[1]:
        raise ValueError(
            f"Target index {target_idx} is unavailable for "
            f"labels with shape {labels_all.shape}"
        )

    target = labels_all[:, target_idx]

    return target_idx, target


In [80]:
def rolling_outer_split(n_samples, train_size, test_size, step_size):
    splits = []
    start = 0

    while start + train_size + test_size <= n_samples:
        train_idx = np.arange(start, start + train_size, dtype=int)
        test_idx = np.arange(start + train_size, start + train_size + test_size, dtype=int)
        splits.append((train_idx, test_idx))
        start += step_size

    return splits


def rolling_inner_split(train_indices, val_size, step_size):
    train_indices = np.asarray(train_indices, dtype=int)
    splits = []
    start = int(train_indices[0])
    end = int(train_indices[-1]) + 1

    while start + val_size <= end:
        inner_train = np.arange(int(train_indices[0]), start, dtype=int)
        val_idx = np.arange(start, start + val_size, dtype=int)

        if len(inner_train) > 0:
            splits.append((inner_train, val_idx))

        start += step_size

    return splits


def safe_corr(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float32).reshape(-1)

    if len(y_true) < 2 or len(y_pred) < 2:
        return np.nan
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    return float(pearsonr(y_true, y_pred)[0])


def concat_result_field(results, key):
    return np.concatenate([np.asarray(item[key]).reshape(-1) for item in results], axis=0)


def summarize_result_list(results):
    y_true = concat_result_field(results, "y_true")
    y_pred = concat_result_field(results, "y_pred")
    y_train_mean = concat_result_field(results, "y_train_mean")

    return {
        "n": int(len(y_true)),
        "mse": float(mean_squared_error(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "corr": safe_corr(y_true, y_pred),
        "train_mean_corr": safe_corr(y_true, y_train_mean),
        "train_mean_mse": float(mean_squared_error(y_true, y_train_mean)),
        "y_true_all": y_true,
        "y_pred_all": y_pred,
        "y_train_mean_all": y_train_mean,
    }


def build_fold_result(model_name, fold_idx, train_idx, test_idx, y_train, y_true, y_pred, metadata=None):
    return {
        "model": model_name,
        "fold": int(fold_idx),
        "train_idx": np.asarray(train_idx, dtype=int),
        "test_idx": np.asarray(test_idx, dtype=int),
        "y_train": np.asarray(y_train, dtype=np.float32).reshape(-1),
        "y_train_mean": np.full(len(y_true), float(np.mean(y_train)), dtype=np.float32),
        "y_true": np.asarray(y_true, dtype=np.float32).reshape(-1),
        "y_pred": np.asarray(y_pred, dtype=np.float32).reshape(-1),
        "metadata": metadata or {},
    }


In [81]:
def _normalize_ts(x):
    x = np.asarray(x, dtype=np.float32)

    if x.ndim == 1:
        x = x[:, None]

    while x.ndim > 2 and x.shape[0] == 1:
        x = x[0]

    if x.ndim != 2:
        raise ValueError(
            f"Unexpected time-series shape: {x.shape}"
        )

    return x


def _normalize_sequence_array(x):
    if isinstance(x, list):
        return np.asarray(x, dtype=object)

    if isinstance(x, np.ndarray) and x.dtype == object:
        return np.asarray(list(x), dtype=object)

    if isinstance(x, np.ndarray) and x.ndim >= 2:
        return np.asarray(
            [sample for sample in x],
            dtype=object,
        )

    raise ValueError(
        f"Unsupported sequence container: "
        f"type={type(x)}, "
        f"shape={getattr(x, 'shape', None)}"
    )


def _infer_feat_dim(stream_list):
    stream_list = _normalize_sequence_array(
        stream_list
    )

    feat_dim = 0

    for x in stream_list:
        if x is None:
            continue

        x = np.asarray(x)

        if x.size == 0:
            continue

        x = _normalize_ts(x)
        feat_dim = max(
            feat_dim,
            int(x.shape[1]),
        )

    return feat_dim


def _sequence_feature_summary(
    x,
    expected_dim,
    include_last=False,
    include_slope=False,
    include_length=False,
    include_missing=False,
):
    n_base_statistics = 5

    def _empty_summary():
        parts = [
            np.full(
                expected_dim * n_base_statistics,
                np.nan,
                dtype=np.float32,
            )
        ]

        if include_last:
            parts.append(
                np.full(
                    expected_dim,
                    np.nan,
                    dtype=np.float32,
                )
            )

        if include_slope:
            parts.append(
                np.full(
                    expected_dim,
                    np.nan,
                    dtype=np.float32,
                )
            )

        if include_length:
            parts.append(
                np.array(
                    [0.0],
                    dtype=np.float32,
                )
            )

        if include_missing:
            parts.append(
                np.array(
                    [1.0],
                    dtype=np.float32,
                )
            )

        return np.concatenate(parts)

    if expected_dim == 0:
        extras = []

        if include_length:
            extras.append(0.0)

        if include_missing:
            extras.append(1.0)

        return np.asarray(
            extras,
            dtype=np.float32,
        )

    if x is None:
        return _empty_summary()

    x = _normalize_ts(x)

    if x.shape[0] == 0:
        return _empty_summary()

    if x.shape[1] < expected_dim:
        pad = expected_dim - x.shape[1]

        x = np.pad(
            x,
            ((0, 0), (0, pad)),
            mode="constant",
            constant_values=np.nan,
        )

    elif x.shape[1] > expected_dim:
        x = x[:, :expected_dim]

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore",
            category=RuntimeWarning,
        )

        mean = np.nanmean(x, axis=0)
        std = np.nanstd(x, axis=0)
        minv = np.nanmin(x, axis=0)
        maxv = np.nanmax(x, axis=0)
        median = np.nanmedian(x, axis=0)

    parts = [
        mean,
        std,
        minv,
        maxv,
        median,
    ]

    if include_last:
        parts.append(x[-1])

    if include_slope:
        if x.shape[0] > 1:
            slope = (
                x[-1] - x[0]
            ) / (x.shape[0] - 1)
        else:
            slope = np.zeros(
                expected_dim,
                dtype=np.float32,
            )

        parts.append(slope)

    if include_length:
        parts.append(
            np.array(
                [float(x.shape[0])],
                dtype=np.float32,
            )
        )

    if include_missing:
        parts.append(
            np.array(
                [float(np.isnan(x).mean())],
                dtype=np.float32,
            )
        )

    return np.concatenate(
        parts
    ).astype(np.float32)


def build_combined_summary_features(bundle):
    """
    For each combined sensor variable, calculate:
    mean, standard deviation, minimum, maximum, and median.

    Also include the sequence length and overall missing-value proportion.
    Missing summary values remain NaN and are imputed within each
    training fold.
    """
    combined = _normalize_sequence_array(
        bundle["combined"]
    )

    combined_dim = _infer_feat_dim(
        combined
    )

    rows = [
        _sequence_feature_summary(
            sample,
            expected_dim=combined_dim,
            include_last=False,
            include_slope=False,
            include_length=True,
            include_missing=True,
        )
        for sample in combined
    ]

    X = np.asarray(
        rows,
        dtype=np.float32,
    )

    # Preserve NaNs for the fold-specific SimpleImputer.
    X[np.isposinf(X)] = np.nan
    X[np.isneginf(X)] = np.nan

    return X

In [82]:
def ensure_2d_feature_matrix(X):
    X = np.asarray(X, dtype=np.float32)

    if X.ndim == 1:
        X = X[:, None]

    elif X.ndim > 2:
        X = X.reshape(X.shape[0], -1)

    # Preserve missing values for fold-specific imputation.
    X[np.isposinf(X)] = np.nan
    X[np.isneginf(X)] = np.nan

    return X


def build_rf_matrices(X_train, X_test):
    X_train = ensure_2d_feature_matrix(X_train)
    X_test = ensure_2d_feature_matrix(X_test)

    imputer = SimpleImputer(
        strategy="median",
        keep_empty_features=True,
    )

    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)

    return X_train_imp, X_test_imp, imputer


def choose_rf_params(
    X_train,
    y_train,
    inner_cfg,
    rf_grid,
):
    X_train = ensure_2d_feature_matrix(X_train)

    y_train = np.asarray(
        y_train,
        dtype=np.float32,
    ).reshape(-1)

    inner_splits = rolling_inner_split(
        np.arange(len(y_train), dtype=int),
        **inner_cfg,
    )

    if not inner_splits:
        return dict(rf_grid[0])

    best_score = np.inf
    best_params = dict(rf_grid[0])

    for params in rf_grid:
        fold_mses = []

        for inner_train_idx, val_idx in inner_splits:
            X_inner_train = X_train[inner_train_idx]
            y_inner_train = y_train[inner_train_idx]

            X_val = X_train[val_idx]
            y_val = y_train[val_idx]

            X_inner_train_imp, X_val_imp, _ = build_rf_matrices(
                X_inner_train,
                X_val,
            )

            model = RandomForestRegressor(
                random_state=42,
                n_jobs=-1,
                **params,
            )

            model.fit(
                X_inner_train_imp,
                y_inner_train,
            )

            val_pred = model.predict(X_val_imp)

            fold_mses.append(
                mean_squared_error(
                    y_val,
                    val_pred,
                )
            )

        mean_mse = float(np.mean(fold_mses))

        if mean_mse < best_score:
            best_score = mean_mse
            best_params = dict(params)

    return best_params


def resolve_pca_n_components(
    requested_n_components,
    X_train,
):
    X_train = ensure_2d_feature_matrix(X_train)
    n_samples, n_features = X_train.shape

    if n_samples <= 1:
        return 1

    max_components = min(
        n_features,
        n_samples - 1,
    )

    return max(
        1,
        min(int(requested_n_components), max_components),
    )


def build_pca_rf_matrices(
    X_train,
    X_test,
    n_components,
):
    X_train = ensure_2d_feature_matrix(X_train)
    X_test = ensure_2d_feature_matrix(X_test)

    X_train_imp, X_test_imp, imputer = build_rf_matrices(
        X_train,
        X_test,
    )

    fitted_n_components = resolve_pca_n_components(
        n_components,
        X_train_imp,
    )

    pca = PCA(
        n_components=fitted_n_components,
        random_state=42,
    )

    X_train_pca = pca.fit_transform(X_train_imp)
    X_test_pca = pca.transform(X_test_imp)

    return X_train_pca, X_test_pca, imputer, pca


def choose_pca_rf_params(
    X_train,
    y_train,
    inner_cfg,
    rf_grid,
    pca_components_grid,
):
    X_train = ensure_2d_feature_matrix(X_train)

    y_train = np.asarray(
        y_train,
        dtype=np.float32,
    ).reshape(-1)

    requested_components = [
        int(value)
        for value in pca_components_grid
        if int(value) > 0
    ]

    if not requested_components:
        raise ValueError(
            "Set RUN_CONFIG['embedding_pca_components'] "
            "to at least one positive integer."
        )

    inner_splits = rolling_inner_split(
        np.arange(len(y_train), dtype=int),
        **inner_cfg,
    )

    default_n_components = resolve_pca_n_components(
        requested_components[0],
        X_train,
    )

    if not inner_splits:
        return {
            "best_params": dict(rf_grid[0]),
            "pca_n_components": default_n_components,
        }

    best_score = np.inf
    best_params = dict(rf_grid[0])
    best_n_components = default_n_components

    for requested_n_components in requested_components:
        fold_mses = {idx: [] for idx in range(len(rf_grid))}

        for inner_train_idx, val_idx in inner_splits:
            X_inner_train = X_train[inner_train_idx]
            y_inner_train = y_train[inner_train_idx]

            X_val = X_train[val_idx]
            y_val = y_train[val_idx]

            X_inner_train_pca, X_val_pca, _, pca = build_pca_rf_matrices(
                X_inner_train,
                X_val,
                n_components=requested_n_components,
            )

            fitted_n_components = int(pca.n_components_)

            for param_idx, params in enumerate(rf_grid):
                model = RandomForestRegressor(
                    random_state=42,
                    n_jobs=-1,
                    **params,
                )

                model.fit(
                    X_inner_train_pca,
                    y_inner_train,
                )

                val_pred = model.predict(X_val_pca)

                fold_mses[param_idx].append(
                    (
                        fitted_n_components,
                        mean_squared_error(
                            y_val,
                            val_pred,
                        ),
                    )
                )

        for param_idx, params in enumerate(rf_grid):
            param_fold_mses = fold_mses[param_idx]

            if not param_fold_mses:
                continue

            fitted_components = int(
                np.median([item[0] for item in param_fold_mses])
            )
            mean_mse = float(
                np.mean([item[1] for item in param_fold_mses])
            )

            if mean_mse < best_score:
                best_score = mean_mse
                best_params = dict(params)
                best_n_components = fitted_components

    return {
        "best_params": best_params,
        "pca_n_components": best_n_components,
    }


In [83]:
def run_train_mean_baseline(target, outer_cfg):
    results = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        rolling_outer_split(len(target), **outer_cfg)
    ):
        y_train = target[train_idx]
        y_true = target[test_idx]

        y_pred = np.full(
            len(test_idx),
            float(np.mean(y_train)),
            dtype=np.float32,
        )

        results.append(
            build_fold_result(
                model_name="Train Mean",
                fold_idx=fold_idx,
                train_idx=train_idx,
                test_idx=test_idx,
                y_train=y_train,
                y_true=y_true,
                y_pred=y_pred,
            )
        )

    return results


def run_last_observation_baseline(target, outer_cfg):
    results = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        rolling_outer_split(len(target), **outer_cfg)
    ):
        y_train = target[train_idx]
        y_true = target[test_idx]

        y_pred = np.full(
            len(test_idx),
            float(y_train[-1]),
            dtype=np.float32,
        )

        results.append(
            build_fold_result(
                model_name="Last Observation",
                fold_idx=fold_idx,
                train_idx=train_idx,
                test_idx=test_idx,
                y_train=y_train,
                y_true=y_true,
                y_pred=y_pred,
            )
        )

    return results


EMBEDDING_RF_MODEL_SPECS = {
    "Chronos Embeddings REG + RF": {
        "strategy": "reg",
    },
    "Chronos Embeddings Mean + RF": {
        "strategy": "mean",
    },
    "Chronos Embeddings Flatten PCA + RF": {
        "strategy": "flatten_pca",
    },
}


def run_rf_only_baseline(
    target,
    rf_features,
    outer_cfg,
    inner_cfg,
    rf_grid,
):
    results = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        rolling_outer_split(len(target), **outer_cfg)
    ):
        y_train = target[train_idx]
        y_true = target[test_idx]

        X_train = rf_features[train_idx]
        X_test = rf_features[test_idx]

        best_params = choose_rf_params(
            X_train=X_train,
            y_train=y_train,
            inner_cfg=inner_cfg,
            rf_grid=rf_grid,
        )

        X_train_imp, X_test_imp, _ = build_rf_matrices(
            X_train,
            X_test,
        )

        model = RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
            **best_params,
        )

        model.fit(X_train_imp, y_train)
        y_pred = model.predict(X_test_imp)

        results.append(
            build_fold_result(
                model_name="RF Only",
                fold_idx=fold_idx,
                train_idx=train_idx,
                test_idx=test_idx,
                y_train=y_train,
                y_true=y_true,
                y_pred=y_pred,
                metadata={
                    "best_params": best_params,
                },
            )
        )

    return results


def run_precomputed_embedding_rf_baseline(
    model_name,
    target,
    outer_cfg,
    inner_cfg,
    rf_grid,
    participant_id,
    embedding_root,
    pca_components_grid,
):
    if model_name not in EMBEDDING_RF_MODEL_SPECS:
        raise ValueError(
            f"Unknown embedding RF model: {model_name}"
        )

    strategy = EMBEDDING_RF_MODEL_SPECS[model_name]["strategy"]
    results = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        rolling_outer_split(len(target), **outer_cfg)
    ):
        y_train = target[train_idx]
        y_true = target[test_idx]

        train_tensor = load_participant_embedding_tensor(
            participant_id,
            outer_fold_idx=fold_idx,
            split_prefix="outer_train",
            embedding_root=embedding_root,
            expected_n=len(train_idx),
        )

        test_tensor = load_participant_embedding_tensor(
            participant_id,
            outer_fold_idx=fold_idx,
            split_prefix="outer_test",
            embedding_root=embedding_root,
            expected_n=len(test_idx),
        )

        if strategy == "flatten_pca":
            X_train = pool_precomputed_embeddings(
                train_tensor,
                strategy="flatten",
            )
            X_test = pool_precomputed_embeddings(
                test_tensor,
                strategy="flatten",
            )

            selection = choose_pca_rf_params(
                X_train=X_train,
                y_train=y_train,
                inner_cfg=inner_cfg,
                rf_grid=rf_grid,
                pca_components_grid=pca_components_grid,
            )

            best_params = selection["best_params"]

            X_train_final, X_test_final, _, pca = build_pca_rf_matrices(
                X_train,
                X_test,
                n_components=selection["pca_n_components"],
            )

            metadata = {
                "best_params": best_params,
                "embedding_strategy": strategy,
                "pca_n_components": int(pca.n_components_),
                "pca_explained_variance_ratio_sum": float(
                    np.sum(pca.explained_variance_ratio_)
                ),
            }

        else:
            X_train = pool_precomputed_embeddings(
                train_tensor,
                strategy=strategy,
            )
            X_test = pool_precomputed_embeddings(
                test_tensor,
                strategy=strategy,
            )

            best_params = choose_rf_params(
                X_train=X_train,
                y_train=y_train,
                inner_cfg=inner_cfg,
                rf_grid=rf_grid,
            )

            X_train_final, X_test_final, _ = build_rf_matrices(
                X_train,
                X_test,
            )

            metadata = {
                "best_params": best_params,
                "embedding_strategy": strategy,
            }

        model = RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
            **best_params,
        )

        model.fit(X_train_final, y_train)
        y_pred = model.predict(X_test_final)

        results.append(
            build_fold_result(
                model_name=model_name,
                fold_idx=fold_idx,
                train_idx=train_idx,
                test_idx=test_idx,
                y_train=y_train,
                y_true=y_true,
                y_pred=y_pred,
                metadata=metadata,
            )
        )

    return results


def get_participant_ids(run_config):
    if run_config.get("participant_ids"):
        return [
            str(participant_id)
            for participant_id in run_config["participant_ids"]
        ]

    if run_config.get("participant_id") is not None:
        return [
            str(run_config["participant_id"])
        ]

    raise ValueError(
        "Set RUN_CONFIG['participant_ids'] "
        "or RUN_CONFIG['participant_id']."
    )


def run_baseline_suite_for_participant(
    run_config,
    participant_id,
):
    participant_id = str(participant_id)

    target_names = list(
        run_config["target_names"]
    )

    model_names = list(
        run_config["model_names"]
    )

    outer_cfg = dict(
        run_config["outer_cfg"]
    )

    inner_cfg = dict(
        run_config["inner_cfg"]
    )

    rf_grid = list(
        run_config["rf_grid"]
    )

    embedding_root = run_config.get(
        "precomputed_embedding_root",
        PRECOMPUTED_EMBEDDING_ROOT,
    )

    pca_components_grid = list(
        run_config.get(
            "embedding_pca_components",
            [8, 16, 32, 64],
        )
    )

    bundle = load_participant_bundle(
        participant_id,
        drop_first_sample=run_config.get(
            "drop_first_sample",
            True,
        ),
    )

    rf_features = None

    if "RF Only" in model_names:
        rf_features = build_combined_summary_features(
            bundle
        )

    metric_rows = []

    for target_name in target_names:
        _, target = select_target(
            bundle,
            target_name,
        )

        for model_name in model_names:
            print(
                f"Running participant={participant_id}, "
                f"target={target_name}, "
                f"model={model_name}"
            )

            if model_name == "Train Mean":
                results = run_train_mean_baseline(
                    target,
                    outer_cfg,
                )

            elif model_name == "Last Observation":
                results = run_last_observation_baseline(
                    target,
                    outer_cfg,
                )

            elif model_name == "RF Only":
                results = run_rf_only_baseline(
                    target=target,
                    rf_features=rf_features,
                    outer_cfg=outer_cfg,
                    inner_cfg=inner_cfg,
                    rf_grid=rf_grid,
                )

            elif model_name in EMBEDDING_RF_MODEL_SPECS:
                results = run_precomputed_embedding_rf_baseline(
                    model_name=model_name,
                    target=target,
                    outer_cfg=outer_cfg,
                    inner_cfg=inner_cfg,
                    rf_grid=rf_grid,
                    participant_id=participant_id,
                    embedding_root=embedding_root,
                    pca_components_grid=pca_components_grid,
                )

            else:
                raise ValueError(
                    f"Unknown model name: {model_name}."
                )

            if not results:
                print(
                    f"No valid outer folds for "
                    f"participant={participant_id}, "
                    f"target={target_name}, "
                    f"model={model_name}"
                )
                continue

            metrics = summarize_result_list(
                results
            )

            metric_rows.append(
                {
                    "participant_id": participant_id,
                    "target": target_name,
                    "model": model_name,
                    "n": metrics["n"],
                    "mse": metrics["mse"],
                    "mae": metrics["mae"],
                    "r2": metrics["r2"],
                    "corr": metrics["corr"],
                    "train_mean_corr": metrics[
                        "train_mean_corr"
                    ],
                    "train_mean_mse": metrics[
                        "train_mean_mse"
                    ],
                }
            )

    return pd.DataFrame(metric_rows)


def normalize_participant_id(value):
    raw = str(value).strip()

    if raw.isdigit():
        return f"{int(raw):05d}"

    return raw


def finalize_metrics_df(
    metrics_df,
    target_names,
    model_names,
):
    metrics_df = metrics_df.copy()
    metrics_df["participant_id"] = metrics_df[
        "participant_id"
    ].map(normalize_participant_id)

    metrics_df["target"] = pd.Categorical(
        metrics_df["target"],
        categories=target_names,
        ordered=True,
    )

    metrics_df["model"] = pd.Categorical(
        metrics_df["model"],
        categories=model_names,
        ordered=True,
    )

    metrics_df = (
        metrics_df
        .sort_values(
            [
                "participant_id",
                "target",
                "model",
            ]
        )
        .reset_index(drop=True)
    )

    return metrics_df


def load_progress_checkpoint(
    output_root,
    filename,
    target_names,
    model_names,
):
    path = Path(output_root) / filename

    if not path.exists():
        return pd.DataFrame(), set()

    checkpoint_df = pd.read_csv(
        path,
        dtype={"participant_id": str},
    )

    if checkpoint_df.empty:
        return checkpoint_df, set()

    required_columns = {
        "participant_id",
        "target",
        "model",
    }

    if not required_columns.issubset(checkpoint_df.columns):
        print(
            f"Ignoring progress checkpoint with unexpected columns: {path}"
        )
        return pd.DataFrame(), set()

    checkpoint_df["participant_id"] = checkpoint_df[
        "participant_id"
    ].map(normalize_participant_id)

    checkpoint_df = checkpoint_df[
        checkpoint_df["target"].isin(target_names)
        & checkpoint_df["model"].isin(model_names)
    ].copy()

    if checkpoint_df.empty:
        return checkpoint_df, set()

    expected_rows_per_participant = (
        len(target_names) * len(model_names)
    )

    participant_row_counts = checkpoint_df.groupby(
        "participant_id"
    ).size()

    completed_participants = set(
        participant_row_counts[
            participant_row_counts >= expected_rows_per_participant
        ].index.tolist()
    )

    checkpoint_df = checkpoint_df[
        checkpoint_df["participant_id"].isin(completed_participants)
    ].copy()

    if checkpoint_df.empty:
        return checkpoint_df, set()

    checkpoint_df = finalize_metrics_df(
        checkpoint_df,
        target_names=target_names,
        model_names=model_names,
    )

    return checkpoint_df, completed_participants


def run_baseline_suite(run_config):
    participant_ids = [
        normalize_participant_id(participant_id)
        for participant_id in get_participant_ids(run_config)
    ]

    target_names = list(
        run_config["target_names"]
    )

    model_names = list(
        run_config["model_names"]
    )

    skip_missing = run_config.get(
        "skip_missing_participants",
        False,
    )

    save_progress = run_config.get(
        "save_progress_after_each_participant",
        True,
    )

    progress_filename = run_config.get(
        "progress_filename",
        "baseline_results_in_progress.csv",
    )

    resume_from_progress = run_config.get(
        "resume_from_progress_checkpoint",
        True,
    )

    all_metrics = []
    completed_participants = set()

    if save_progress and resume_from_progress:
        checkpoint_df, completed_participants = load_progress_checkpoint(
            output_root=OUTPUT_ROOT,
            filename=progress_filename,
            target_names=target_names,
            model_names=model_names,
        )

        if not checkpoint_df.empty:
            all_metrics.append(checkpoint_df)
            print(
                f"Resuming from checkpoint for participants: "
                f"{sorted(completed_participants)}"
            )

    for participant_id in participant_ids:
        if participant_id in completed_participants:
            print(
                f"Skipping already completed participant {participant_id}."
            )
            continue

        try:
            participant_metrics = (
                run_baseline_suite_for_participant(
                    run_config,
                    participant_id,
                )
            )

        except FileNotFoundError as exc:
            if skip_missing:
                print(
                    f"Skipping participant "
                    f"{participant_id}: {exc}"
                )
                continue

            raise

        if not participant_metrics.empty:
            all_metrics.append(
                participant_metrics
            )

            if save_progress:
                checkpoint_df = finalize_metrics_df(
                    pd.concat(
                        all_metrics,
                        ignore_index=True,
                    ),
                    target_names=target_names,
                    model_names=model_names,
                )

                save_baseline_results(
                    checkpoint_df,
                    filename=progress_filename,
                )

                print(
                    f"Saved progress checkpoint after participant "
                    f"{participant_id}."
                )

    if not all_metrics:
        raise ValueError(
            "No participant results were produced."
        )

    metrics_df = finalize_metrics_df(
        pd.concat(
            all_metrics,
            ignore_index=True,
        ),
        target_names=target_names,
        model_names=model_names,
    )

    return metrics_df


def save_baseline_results(
    metrics_df,
    output_root=OUTPUT_ROOT,
    filename="baseline_results.csv",
):
    output_root = Path(output_root)
    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path = output_root / filename

    metrics_df.to_csv(
        output_path,
        index=False,
    )

    print(
        f"Saved full baseline result table to: "
        f"{output_path}"
    )

    return output_path

## Run Configuration

Set `participant_ids` to the list you want to run. The notebook now supports single-participant and multi-participant runs, and the aggregate summary is computed across all successfully run participants.


In [84]:
MODEL_ORDER = [
    "Train Mean",
    "Last Observation",
    "RF Only",
    "Chronos Embeddings REG + RF",
    "Chronos Embeddings Mean + RF",
    "Chronos Embeddings Flatten PCA + RF",
]

RUN_CONFIG = {
    "participant_ids": [
        "00092",
        "00011",
        "00012",
        "00095",
        "00179",
        "00368",
        "00424",
        "00477",
        "00485",
        "00551",
    ],
    "target_names": [
        "Negative",
        "Cognitive",
        "Positive",
    ],
    "model_names": MODEL_ORDER,
    "outer_cfg": {
        "train_size": 135,
        "test_size": 3,
        "step_size": 3,
    },
    "inner_cfg": {
        "val_size": 21,
        "step_size": 114,
    },
    "rf_grid": DEFAULT_RF_GRID,
    "drop_first_sample": True,
    "skip_missing_participants": True,
    "precomputed_embedding_root": PRECOMPUTED_EMBEDDING_ROOT,
    "embedding_pca_components": [8, 16, 32, 64],
    "save_progress_after_each_participant": True,
    "progress_filename": "baseline_results_in_progress.csv",
    "resume_from_progress_checkpoint": True,
}

RUN_CONFIG

{'participant_ids': ['00092',
  '00011',
  '00012',
  '00095',
  '00179',
  '00368',
  '00424',
  '00477',
  '00485',
  '00551'],
 'target_names': ['Negative', 'Cognitive', 'Positive'],
 'model_names': ['Train Mean',
  'Last Observation',
  'RF Only',
  'Chronos Embeddings REG + RF',
  'Chronos Embeddings Mean + RF',
  'Chronos Embeddings Flatten PCA + RF'],
 'outer_cfg': {'train_size': 135, 'test_size': 3, 'step_size': 3},
 'inner_cfg': {'val_size': 21, 'step_size': 114},
 'rf_grid': [{'n_estimators': 300,
   'max_depth': 4,
   'min_samples_leaf': 4,
   'max_features': 'sqrt'},
  {'n_estimators': 300,
   'max_depth': 8,
   'min_samples_leaf': 4,
   'max_features': 'sqrt'},
  {'n_estimators': 400,
   'max_depth': None,
   'min_samples_leaf': 8,
   'max_features': 0.2}],
 'drop_first_sample': True,
 'skip_missing_participants': True,
 'precomputed_embedding_root': PosixPath('/Users/f007qrc/Library/CloudStorage/GoogleDrive-anna.m.langener@dartmouth.edu/.shortcut-targets-by-id/1_T-CzrkNtF

In [89]:
metrics_df = run_baseline_suite(RUN_CONFIG)

display(metrics_df)

output_path = save_baseline_results(
    metrics_df,
)

print(f"Saved outputs to: {output_path}")

Resuming from checkpoint for participants: ['00011', '00012', '00092', '00095', '00179', '00368', '00424', '00477', '00485', '00551']
Skipping already completed participant 00092.
Skipping already completed participant 00011.
Skipping already completed participant 00012.
Skipping already completed participant 00095.
Skipping already completed participant 00179.
Skipping already completed participant 00368.
Skipping already completed participant 00424.
Skipping already completed participant 00477.
Skipping already completed participant 00485.
Skipping already completed participant 00551.


,participant_id,target,model,n,mse,mae,r2,corr,train_mean_corr,train_mean_mse
0,00011,Negative,Train Mean,135,0.021386,0.122419,-0.026573,0.152781,0.152781,0.021386
1,00011,Negative,Last Observation,135,0.028193,0.136296,-0.353307,0.322771,0.152781,0.021386
2,00011,Negative,RF Only,135,0.022182,0.122802,-0.064766,0.079894,0.152781,0.021386
3,00011,Negative,Chronos Embeddings REG + RF,135,0.022669,0.124886,-0.088145,0.048040,0.152781,0.021386
4,00011,Negative,Chronos Embeddings Mean + RF,135,0.022113,0.123115,-0.061465,0.106803,0.152781,0.021386
...,...,...,...,...,...,...,...,...,...,...
175,00551,Positive,Last Observation,105,0.025546,0.112667,-0.245955,0.261233,-0.258706,0.021396
176,00551,Positive,RF Only,105,0.021841,0.111390,-0.065276,-0.104556,-0.258706,0.021396
177,00551,Positive,Chronos Embeddings REG + RF,105,0.020827,0.108313,-0.015790,0.091982,-0.258706,0.021396
178,00551,Positive,Chronos Embeddings Mean + RF,105,0.020414,0.107618,0.004314,0.125767,-0.258706,0.021396


Saved full baseline result table to: /Users/f007qrc/Library/CloudStorage/GoogleDrive-anna.m.langener@dartmouth.edu/My Drive/Darmouth Drive/6_PersonalizedSensingSchizophrenia/baseline_outputs/baseline_results.csv
Saved outputs to: /Users/f007qrc/Library/CloudStorage/GoogleDrive-anna.m.langener@dartmouth.edu/My Drive/Darmouth Drive/6_PersonalizedSensingSchizophrenia/baseline_outputs/baseline_results.csv


## Compare with main results

In [2]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


CORRELATION_METHOD = "pearson"


# =============================================================================
# Paths
# =============================================================================

BASE_DIR = Path(
    "/Users/f007qrc/Library/CloudStorage/"
    "GoogleDrive-anna.m.langener@dartmouth.edu/"
    ".shortcut-targets-by-id/"
    "1_T-CzrkNtF9QAhex6qrEOA0ZQIL7g5u5/"
    "SensingIndividualized"
)

RESULTS_ROOT = BASE_DIR / "results_true_pred"
DATA_DIR = BASE_DIR / "deeplearning_data" / "pred"

BASELINE_RESULTS_PATH = (
 "/Users/f007qrc/Library/CloudStorage/GoogleDrive-anna.m.langener@dartmouth.edu/My Drive/Darmouth Drive/6_PersonalizedSensingSchizophrenia/baseline_outputs/baseline_results.csv"

)

COMBINED_RESULTS_SAVE_PATH = (
    RESULTS_ROOT
    / "model_and_baseline_results.csv"
)

OUTER_CFG = {
    "train_size": 135,
    "test_size": 3,
    "step_size": 3,
}

ALPHA = 0.05


# =============================================================================
# Helper functions
# =============================================================================

def safe_torch_load(path):
    """
    Load a PyTorch file while remaining compatible with newer and older
    PyTorch versions.
    """
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )


def pooled_correlation(
    y_true,
    y_pred,
    method=CORRELATION_METHOD,
    alpha=ALPHA,
):
    """
    Compute a pooled correlation across all test observations.
    """
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) < 3:
        return np.nan, np.nan, False

    if np.ptp(y_true) == 0 or np.ptp(y_pred) == 0:
        return np.nan, np.nan, False

    method = method.lower()

    if method == "spearman":
        correlation, p_value = spearmanr(
            y_true,
            y_pred,
        )

    elif method == "pearson":
        correlation, p_value = pearsonr(
            y_true,
            y_pred,
        )

    else:
        raise ValueError(
            "method must be either 'spearman' or 'pearson'. "
            f"Received: {method!r}"
        )

    if (
        not np.isfinite(correlation)
        or not np.isfinite(p_value)
    ):
        return np.nan, np.nan, False

    significant = bool(
        p_value < alpha
    )

    return (
        float(correlation),
        float(p_value),
        significant,
    )


def get_expected_n_splits(
    participant_id,
    data_dir=DATA_DIR,
    outer_cfg=OUTER_CFG,
):
    """
    Compute the expected number of outer splits from the participant .npz file.

    This mirrors the preprocessing step in which the first questionnaire
    observation is removed.
    """
    file_path = (
        data_dir
        / f"participant_{participant_id}_4h.npz"
    )

    if not file_path.exists():
        return np.nan

    loaded = np.load(
        file_path,
        allow_pickle=True,
    )

    n_samples = len(
        loaded["labels"]
    )

    # The model pipeline removes the first observation.
    n_samples -= 1

    train_size = outer_cfg["train_size"]
    test_size = outer_cfg["test_size"]
    step_size = outer_cfg["step_size"]

    n_splits = 0
    start = 0

    while (
        start
        + train_size
        + test_size
        <= n_samples
    ):
        n_splits += 1
        start += step_size

    return n_splits


def parse_result_filename(path):
    """
    Expected filenames:
        results_00092_negative.pth
        results_00011_cognitive.pth
        results_00011_positive.pth
    """
    match = re.match(
        r"results_(\d+)_(negative|cognitive|positive)\.pth$",
        path.name,
    )

    if match is None:
        return None, None

    participant_id = match.group(1)
    outcome = match.group(2)

    return participant_id, outcome


def make_error_row(
    path,
    participant_id,
    outcome,
    variant_folder,
    error_message,
):
    """
    Create a consistent summary row for files that could not be processed.
    """
    return {
        "variant_folder": variant_folder,
        "participant_id": participant_id,
        "outcome": outcome,
        "n_splits": np.nan,
        "expected_splits": get_expected_n_splits(
            participant_id
        ),
        "complete": False,
        "n_test_obs": np.nan,
        "pooled_corr": np.nan,
        "pooled_corr_p": np.nan,
        "corr_significant": False,
        "corr_significance": "NA",
        "pooled_mse": np.nan,
        "pooled_mae": np.nan,
        "pooled_r2": np.nan,
        "mean_split_mse": np.nan,
        "max_outer_i": np.nan,
        "missing_outer_ids": [],
        "file_path": str(path),
        "load_error": str(error_message),
    }


def summarize_result_file(path):
    """
    Load one result file and compute pooled performance metrics.
    """
    participant_id, outcome = parse_result_filename(
        path
    )

    if participant_id is None:
        return None

    variant_folder = path.parent.name

    try:
        results = safe_torch_load(
            path
        )

    except Exception as error:
        return make_error_row(
            path=path,
            participant_id=participant_id,
            outcome=outcome,
            variant_folder=variant_folder,
            error_message=error,
        )

    if not isinstance(results, list):
        return make_error_row(
            path=path,
            participant_id=participant_id,
            outcome=outcome,
            variant_folder=variant_folder,
            error_message=(
                f"Expected list, got {type(results)}"
            ),
        )

    n_splits = len(results)

    expected_splits = get_expected_n_splits(
        participant_id
    )

    y_true_all = []
    y_pred_all = []
    split_mses = []
    outer_ids = []

    for item in results:
        if item is None:
            continue

        y_test = np.asarray(
            item.get(
                "y_test",
                item.get("y_true", []),
            )
        ).reshape(-1)

        y_pred = np.asarray(
            item.get("y_pred", [])
        ).reshape(-1)

        if (
            len(y_test) == 0
            or len(y_pred) == 0
        ):
            continue

        if len(y_test) != len(y_pred):
            continue

        valid_mask = (
            np.isfinite(y_test)
            & np.isfinite(y_pred)
        )

        y_test = y_test[
            valid_mask
        ]

        y_pred = y_pred[
            valid_mask
        ]

        if len(y_test) == 0:
            continue

        y_true_all.append(
            y_test
        )

        y_pred_all.append(
            y_pred
        )

        split_mses.append(
            mean_squared_error(
                y_test,
                y_pred,
            )
        )

        if "outer_i" in item:
            outer_ids.append(
                int(item["outer_i"])
            )

        elif "fold" in item:
            outer_ids.append(
                int(item["fold"])
            )

    if len(y_true_all) > 0:
        y_true_all = np.concatenate(
            y_true_all
        )

        y_pred_all = np.concatenate(
            y_pred_all
        )

        (
            corr,
            corr_p_value,
            corr_significant,
        ) = pooled_correlation(
            y_true_all,
            y_pred_all,
            method=CORRELATION_METHOD,
            alpha=ALPHA,
        )

        pooled_mse = mean_squared_error(
            y_true_all,
            y_pred_all,
        )

        pooled_mae = mean_absolute_error(
            y_true_all,
            y_pred_all,
        )

        pooled_r2 = r2_score(
            y_true_all,
            y_pred_all,
        )

        n_test_obs = len(
            y_true_all
        )

    else:
        corr = np.nan
        corr_p_value = np.nan
        corr_significant = False
        pooled_mse = np.nan
        pooled_mae = np.nan
        pooled_r2 = np.nan
        n_test_obs = 0

    if np.isnan(corr_p_value):
        corr_significance = "NA"

    elif corr_p_value < 0.001:
        corr_significance = "***"

    elif corr_p_value < 0.01:
        corr_significance = "**"

    elif corr_p_value < 0.05:
        corr_significance = "*"

    else:
        corr_significance = "ns"

    if len(split_mses) > 0:
        mean_split_mse = float(
            np.mean(split_mses)
        )

    else:
        mean_split_mse = np.nan

    if len(outer_ids) > 0:
        outer_ids = sorted(
            set(outer_ids)
        )

        max_outer_i = max(
            outer_ids
        )

        missing_outer_ids = sorted(
            set(
                range(max_outer_i + 1)
            )
            - set(outer_ids)
        )

    else:
        missing_outer_ids = []
        max_outer_i = np.nan

    if np.isnan(expected_splits):
        complete = False

    else:
        complete = (
            n_splits
            == expected_splits
        )

    return {
        "variant_folder": variant_folder,
        "participant_id": participant_id,
        "outcome": outcome,
        "n_splits": n_splits,
        "expected_splits": expected_splits,
        "complete": complete,
        "n_test_obs": n_test_obs,
        "pooled_corr": corr,
        "pooled_corr_p": corr_p_value,
        "corr_significant": corr_significant,
        "corr_significance": corr_significance,
        "pooled_mse": pooled_mse,
        "pooled_mae": pooled_mae,
        "pooled_r2": pooled_r2,
        "mean_split_mse": mean_split_mse,
        "max_outer_i": max_outer_i,
        "missing_outer_ids": missing_outer_ids,
        "file_path": str(path),
        "load_error": None,
    }


# =============================================================================
# Summarize Model result files
# =============================================================================

result_files = sorted(
    RESULTS_ROOT.glob(
        "Model*/results_*_*.pth"
    )
)

rows = []

for path in result_files:
    row = summarize_result_file(
        path
    )

    if row is not None:
        rows.append(row)

model_results_df = pd.DataFrame(
    rows
)

if len(model_results_df) == 0:
    print(
        f"No Model result files found in: "
        f"{RESULTS_ROOT}"
    )

else:
    model_results_df = model_results_df.rename(
        columns={
            "variant_folder": "model",
            "outcome": "target",
            "n_test_obs": "n",
            "pooled_corr": "corr",
            "pooled_corr_p": "corr_p",
            "pooled_mse": "mse",
            "pooled_mae": "mae",
            "pooled_r2": "r2",
        }
    )

    model_results_df["participant_id"] = (
        model_results_df["participant_id"]
        .astype(str)
        .str.zfill(5)
    )

    model_results_df["target"] = (
        model_results_df["target"]
        .str.capitalize()
    )

    model_results_df[
        "result_source"
    ] = "Neural model"


# =============================================================================
# Load baseline results
# =============================================================================

# if not BASELINE_RESULTS_PATH.exists():
#     raise FileNotFoundError(
#         f"Baseline results not found: "
#         f"{BASELINE_RESULTS_PATH}"
#     )

baseline_results_df = pd.read_csv(
    BASELINE_RESULTS_PATH,
    dtype={
        "participant_id": str,
    },
)

baseline_results_df["participant_id"] = (
    baseline_results_df["participant_id"]
    .astype(str)
    .str.zfill(5)
)

baseline_results_df["target"] = (
    baseline_results_df["target"]
    .str.capitalize()
)

baseline_results_df[
    "result_source"
] = "Baseline"

if "corr_p" not in baseline_results_df.columns:
    baseline_results_df[
        "corr_p"
    ] = np.nan

if "complete" not in baseline_results_df.columns:
    baseline_results_df[
        "complete"
    ] = True

if "file_path" not in baseline_results_df.columns:
    baseline_results_df[
        "file_path"
    ] = str(
        BASELINE_RESULTS_PATH
    )

if "load_error" not in baseline_results_df.columns:
    baseline_results_df[
        "load_error"
    ] = None


# =============================================================================
# Combine baseline and model results
# =============================================================================

comparison_columns = [
    "participant_id",
    "target",
    "model",
    "result_source",
    "n",
    "corr",
    "corr_p",
    "mse",
    "mae",
    "r2",
    "complete",
    "file_path",
    "load_error",
]


def add_missing_columns(
    dataframe,
    columns,
):
    dataframe = dataframe.copy()

    for column in columns:
        if column not in dataframe.columns:
            dataframe[
                column
            ] = np.nan

    return dataframe[
        columns
    ]


baseline_comparison_df = add_missing_columns(
    baseline_results_df,
    comparison_columns,
)

model_comparison_df = add_missing_columns(
    model_results_df,
    comparison_columns,
)

combined_results_df = pd.concat(
    [
        baseline_comparison_df,
        model_comparison_df,
    ],
    ignore_index=True,
)

combined_results_df["participant_id"] = (
    combined_results_df["participant_id"]
    .astype(str)
    .str.zfill(5)
)

combined_results_df["target"] = pd.Categorical(
    combined_results_df["target"],
    categories=[
        "Negative",
        "Cognitive",
        "Positive",
    ],
    ordered=True,
)

combined_results_df = (
    combined_results_df
    .sort_values(
        by=[
            "model",
            "participant_id",
            "target",
        ]
    )
    .reset_index(drop=True)
)


# =============================================================================
# Save combined table
# =============================================================================

combined_results_df.to_csv(
    COMBINED_RESULTS_SAVE_PATH,
    index=False,
)

print(
    f"Found {len(model_results_df)} "
    "Model result files."
)

print(
    f"Loaded {len(baseline_results_df)} "
    "baseline rows."
)

print(
    f"Saved combined results to: "
    f"{COMBINED_RESULTS_SAVE_PATH}"
)


# =============================================================================
# Compact table
# =============================================================================

compact_cols = [
    "model",
    "result_source",
    "participant_id",
    "target",
    "n",
    "complete",
    "corr",
    "corr_p",
    "mse",
    "mae",
    "r2",
]


display(
    combined_results_df[
        compact_cols
    ]
)


# =============================================================================
# Mean results across participants
# =============================================================================

mean_results_table = (
    combined_results_df
    .groupby(
        [
            "target",
            "model",
        ],
        observed=True,
    )
    .agg(
        n_participants=(
            "participant_id",
            "nunique",
        ),
        mean_corr=(
            "corr",
            "mean",
        ),
        mean_mse=(
            "mse",
            "mean",
        ),
        mean_mae=(
            "mae",
            "mean",
        ),
        mean_r2=(
            "r2",
            "mean",
        ),
    )
    .reset_index()
)

display(
    mean_results_table.style.format(
        {
            "mean_corr": "{:.3f}",
            "mean_mse": "{:.4f}",
            "mean_mae": "{:.4f}",
            "mean_r2": "{:.3f}",
        }
    )
)

Found 270 Model result files.
Loaded 180 baseline rows.
Saved combined results to: /Users/f007qrc/Library/CloudStorage/GoogleDrive-anna.m.langener@dartmouth.edu/.shortcut-targets-by-id/1_T-CzrkNtF9QAhex6qrEOA0ZQIL7g5u5/SensingIndividualized/results_true_pred/model_and_baseline_results.csv


,model,result_source,participant_id,target,n,complete,corr,corr_p,mse,mae,r2
0,Chronos Embeddings Flatten PCA + RF,Baseline,00011,Negative,135,True,0.080838,NaN,0.021815,0.122740,-0.047165
1,Chronos Embeddings Flatten PCA + RF,Baseline,00011,Cognitive,135,True,0.042338,NaN,0.028860,0.139800,-0.007625
2,Chronos Embeddings Flatten PCA + RF,Baseline,00011,Positive,135,True,-0.104571,NaN,0.008328,0.069150,-0.045482
3,Chronos Embeddings Flatten PCA + RF,Baseline,00012,Negative,120,True,-0.076346,NaN,0.106259,0.261404,-0.368502
4,Chronos Embeddings Flatten PCA + RF,Baseline,00012,Cognitive,120,True,0.064352,NaN,0.036025,0.157903,-0.011764
...,...,...,...,...,...,...,...,...,...,...,...
445,Train Mean,Baseline,00485,Cognitive,84,True,0.030522,NaN,0.007078,0.067254,-0.010579
446,Train Mean,Baseline,00485,Positive,84,True,0.103461,NaN,0.007657,0.067929,-0.001462
447,Train Mean,Baseline,00551,Negative,105,True,-0.324082,NaN,0.033057,0.152188,-0.060503
448,Train Mean,Baseline,00551,Cognitive,105,True,-0.019062,NaN,0.036837,0.175529,-0.026493


,target,model,n_participants,mean_corr,mean_mse,mean_mae,mean_r2
0,Negative,Chronos Embeddings Flatten PCA + RF,10,-0.001,0.0374,0.1431,-0.154
1,Negative,Chronos Embeddings Mean + RF,10,0.020,0.0389,0.1495,-0.215
2,Negative,Chronos Embeddings REG + RF,10,0.027,0.0388,0.1499,-0.205
3,Negative,Last Observation,10,0.152,0.0562,0.1475,-0.697
4,Negative,Model1_Frozen_Combined_no_retrain_single,10,0.115,0.0356,0.1300,-0.113
5,Negative,Model2_LoRA_Combined_no_retrain_single,10,0.059,0.0393,0.1421,-0.195
6,Negative,Model3_reg_quantile_no_retrain_single,10,0.058,0.0389,0.1292,-0.174
7,Negative,Model4_Personal60_OtherParticipants40_CheckpointTransfer_no_retrain_single,10,0.120,0.0459,0.1783,-1.466
8,Negative,Model4_Personal60_OtherParticipants40_CheckpointTransfer_positiveO1FCorr_no_retrain_single,10,0.120,0.0461,0.1800,-2.024
9,Negative,Model4_Personal60_OtherParticipants40_CheckpointTransfer_positiveO2FCorr_no_retrain_single,10,0.119,0.0496,0.1860,-2.578
